### Setup Environment & Redis Connection

By default, this notebook connects to the local instance of Redis Stack using the variables in your `.env` file.

In [21]:
import os
import warnings
from dotenv import load_dotenv

# Suppress noisy warnings and load environment variables
warnings.filterwarnings('ignore')
load_dotenv(override=True)

# Define Redis connection
REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
REDIS_PORT = os.getenv("REDIS_PORT", "6379")
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")

if REDIS_PASSWORD:
    REDIS_URL = f"redis://:{REDIS_PASSWORD}@{REDIS_HOST}:{REDIS_PORT}"
else:
    REDIS_URL = f"redis://{REDIS_HOST}:{REDIS_PORT}"

print(f"Connected to Redis at: {REDIS_URL}")

Connected to Redis at: redis://localhost:6379


## Dataset Preparation (PDF Documents)

We will load the financial 10-K document and preprocess it using LangChain's `UnstructuredLoader` and `RecursiveCharacterTextSplitter`.

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_unstructured import UnstructuredLoader

# Load list of pdfs from the resources folder
data_path = "resources/"
docs = [os.path.join(data_path, file) for file in os.listdir(data_path) if file.endswith('.pdf')]
print("Available documents:", docs)

# Pick out the Nike doc for this exercise
doc_path = [doc for doc in docs if "nke" in doc][0]

# Set up the file loader and text splitter (using chunk_overlap=100 to match main.py)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2500, 
    chunk_overlap=100
)

loader = UnstructuredLoader(
    doc_path, 
    chunking_strategy="basic", 
    max_characters=1000000,
    include_orig_elements=False,
    strategy="fast"
)

# Extract, load, and make chunks
chunks = loader.load_and_split(text_splitter)
print(f"Done preprocessing. Created {len(chunks)} chunks from {doc_path}")

Available documents: ['resources/aapl-10k-2023.pdf', 'resources/amzn-10k-2023.pdf', 'resources/jnj-10k-2023.pdf', 'resources/msft-10k-2023.pdf', 'resources/nke-10k-2023.pdf', 'resources/nvd-10k-2023.pdf']


Done preprocessing. Created 182 chunks from resources/nke-10k-2023.pdf


### Initialize Embeddings Model

We will use HuggingFace embeddings (`all-MiniLM-L6-v2`) to convert our text chunks into dense vector representations.

In [23]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

INFO: No device provided, using cpu
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fconfig_sentence_transformers.json=&etag=%22fd1b291129c607e5d49799f87cb219b27f98acdf%22

## Vector Search with LangChain and Redis

Create the Redis vector store instance and index the documents.

In [24]:
from langchain_redis import RedisVectorStore

# Standardized index name to match main.py
index_name = "financial_10k_rag"

# Construct the vector store class from texts and metadata
rds = RedisVectorStore.from_documents(
    chunks,
    embeddings,
    index_name=index_name,
    redis_url=REDIS_URL,
    metadata_schema=[
        {"name": "source", "type": "text"}
    ]
)

# Access underlying redis client to see how many docs have been stored
print(f"Total documents in Redis index: {rds._index.client.dbsize()}")

INFO: Index already exists, not overwriting.


Total documents in Redis index: 364


### Query the Database

Perform similarity search operations on Redis to ensure our vectors are retrieving the correct context.

In [25]:
from redisvl.query.filter import Text

# Basic vector search
results = rds.similarity_search_with_score(query="Profit margins", k=2)

for doc, score in results:
    print(f"Score: {score:.4f} | Source: {doc.metadata.get('source')}")
    print(f"Preview: {doc.page_content[:150]}...\n")

Score: 0.5278 | Source: resources/nke-10k-2023.pdf
Preview: 2023 FORM 10-K 30

Table of Contents

RESULTS OF OPERATIONS

(Dollars in millions, except per share data)

Revenues Cost of sales

Gross profit Gross ...

Score: 0.5278 | Source: resources/nke-10k-2023.pdf
Preview: 2023 FORM 10-K 30

Table of Contents

RESULTS OF OPERATIONS

(Dollars in millions, except per share data)

Revenues Cost of sales

Gross profit Gross ...



## RAG Pipeline Orchestration

Initialize the OpenRouter LLM and build the LangChain retrieval chain.

In [26]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Initialize OpenRouter Chat Model
openrouter_key = os.getenv("OPENROUTER_API_KEY")
model_name = os.getenv("OPENROUTER_MODEL", "apodex/apodex-1.1-mini:free")

llm = ChatOpenAI(
    model=model_name,
    openai_api_key=openrouter_key,
    openai_api_base="https://openrouter.ai/api/v1", # Forces the correct endpoint
    default_headers={
        "HTTP-Referer": "https://github.com/radis-langchain",
        "X-Title": "Financial Document RAG",
    },
    temperature=0.2,
)

# Setup Prompt Template
prompt_template = """
Use the following pieces of context from financial 10-K filings to answer the user question at the end. 
If you don't know the answer, say that you don't know, don't try to make up an answer.

Context:
---------
{context}
---------

Question: {question}
Answer:
"""
prompt = ChatPromptTemplate.from_template(prompt_template)

# Helper to format docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Construct RAG Chain
rag_chain = (
    {
        "context": rds.as_retriever(search_kwargs={"k": 4}) | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

### Ask Questions

Test the RAG pipeline with sample financial queries.

In [27]:
query = "What was Nike's revenue last year compared to this year?"
response = rag_chain.invoke(query)
print(response)

INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"




Based on the 2023 Form 10-K provided, Nike's total revenues were **$51,217 million** for Fiscal 2023 (this year) compared to **$46,710 million** for Fiscal 2022 (last year). This represents a **10% increase** in revenue year-over-year (or 16% excluding currency changes).


### Cleanup

Remove the index and data from Redis once finished.

In [28]:
from redisvl.index import SearchIndex

try:
    idx = SearchIndex.from_existing(
        index_name,
        redis_url=REDIS_URL
    )
    idx.delete()
    print(f"Successfully deleted index: {index_name}")
except Exception as e:
    print(f"Error during cleanup: {e}")

Successfully deleted index: financial_10k_rag
